In [ ]:
# api基础调用
from openai import OpenAI
import os

client=OpenAI(api_key=os.getenv('DEEPSEEK_API_KEY'),
base_url="https://api.deepseek.com"
)

response=client.chat.completions.create(
    model="deepseek-v4-pro",
    messages=[
         {"role":"system","content":"你是可爱的助手，你的名字是小祥，请用可爱的语气和用户聊天,并且不会说废话"},
         {"role":"user","content":"2026年父亲节什么时候"}],
    stream=False
)
print(response.choices[0].message.content)

In [ ]:
#流式输出
from openai import OpenAI
import os

client=OpenAI(api_key=os.getenv('DEEPSEEK_API_KEY'),
base_url="https://api.deepseek.com"
)

response=client.chat.completions.create(
    model="deepseek-v4-pro",
    messages=[
         {"role":"system","content":"你是可爱的助手，你的名字是小祥，请用可爱的语气和用户聊天,并且不会说废话"},
         {"role":"user","content":"今天是几号？"}],
    stream=True,
    temperature=1
)
for chunk in response:
    chunk=chunk.choices[0].delta.content
    if chunk is not None:
        print(chunk,end="",flush=True)

In [ ]:
#function calling
# 1. 定义工具清单
tools=[
    {
        "type":"function",
        "function":{
            "name":"get_weather",
            "description":"获得指定城市的天气情况",
            "parameters":{
                "type":"object",
                "properties":{
                    "city":{"type":"string",
                    "description":"城市名称"}
                },
                "required":["city"]
            }
        }
    }
]
from openai import OpenAI
import os,json
# 用户问题
messages=[{"role": "user", "content": "我想知道合肥今天天气如何，适合出去玩吗？"}]

client=OpenAI(api_key=os.getenv('DEEPSEEK_API_KEY'),
base_url="https://api.deepseek.com"
)
# 3. 第一次调用：模型决定要不要用工具
response=client.chat.completions.create(
    model="deepseek-v4-pro",
    messages=messages,
    stream=False,
    tools=tools,
    tool_choice="auto"
    )
# 4. 检查模型是不是要调用函数
msg=response.choices[0].message
if msg.tool_calls:
    tool_call=msg.tool_calls[0]
    tool_name=tool_call.function.name
    tool_args=json.loads(tool_call.function.arguments)
    print(f"调用了工具：{tool_name}，参数是：{tool_args}")
     # 5. 我们在本地执行“真实的”天气查询（这里模拟）
    if tool_name=="get_weather":
        weather_result=f"{tool_args['city']}的天气情况是晴朗，适合出去玩哦！"
# 6. 把工具执行结果追加到对话里
    messages.append(msg)
    messages.append({"role":"tool","tool_call_id":tool_call.id,"content":weather_result})
# 7. 第二次调用：把工具结果告诉模型，让它生成最终回复
    final_response=client.chat.completions.create(
    model="deepseek-v4-pro",
    messages=messages,
    stream=False
)
    print(final_response.choices[0].message.content)
else:
    print("模型没有调用工具，回复是：",msg.content)


In [ ]:
# json output
from openai import OpenAI
import os

client=OpenAI(api_key=os.getenv('DEEPSEEK_API_KEY'),
base_url="https://api.deepseek.com"
)

response=client.chat.completions.create(
    model="deepseek-v4-pro",
    messages=[
         {"role":"system","content":"你是可爱的助手，你的名字是小祥，请用可爱的语气和用户聊天,并且不会说废话"},
         {"role":"user","content":"2026年父亲节什么时候"}],
    stream=False,
    response_format={"type":"json_object"}
)
print(response.choices[0].message.content)